# Check what training augmentation does to our cassette labels

**Use a CPU runtime and Run all. No model is trained or loaded.**
This diagnostic uses the same 142 training photographs and frozen labels
as the quality-focal experiment. It compares the unchanged source labels
with native letterboxing and with the existing training augmentation.

Three fixed passes produce 426 image presentations. For every original
label row, the notebook records clipping, native box-filter decisions,
source and transformed corner availability, horizontal slot exchange,
and final target geometry. It creates at most six review panels.

A removed target is not automatically a mistake: the transformed object
may be outside the image or too small to identify. A box-area ratio is
not the physical fraction of visible cassette material. Review the
transformed pixels before deciding whether an augmentation should change.
These are new diagnostic draws, not a reconstruction of the random draws
in a historical training run. No evaluation photographs enter this audit.


## 1 Load the fixed source

Existing incompatible source and outputs are preserved. The optional
`BAOMAI_R61_CACHE` setting shares verified source assets with the existing
training notebooks. Set `BAOMAI_AUGMENTATION_AUDIT_BASE` to a fresh folder
for an independent rerun.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
sys.dont_write_bytecode = True

SOURCE_COMMIT = "87b3f1dfd547bf46fef35b171aebe16d832bd1be"
DRIVER_SHA256 = "1bdeb8fb4df6c37ba80c6836f684f8462680373ad0fdc673f039ea31b7b9615e"
assert SOURCE_COMMIT != "0" * 40, "This draft needs its published source pin."
BASE = Path(os.environ.get("BAOMAI_AUGMENTATION_AUDIT_BASE", "/content/baomai-augmentation-audit-r63")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
CACHE = Path(os.environ.get("BAOMAI_R61_CACHE", "/content/baomai-r61-cache")).resolve()
ASSETS = CACHE / "assets"
BASE.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Nonempty source preserved; choose a fresh BAOMAI_AUGMENTATION_AUDIT_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", *['/scripts/', '/metadata/model_card.json', '/models/cassette_lower_yolo26s_pose.yaml', '/data/manifest.json', '/data/labels/', '/data/candidates/front-review-2026-11-26-r37/', '/data/references/sequence-refined-reference-2026-11-18/', '/data/references/visible-front-completion-2026-11-14/', '/data/reviews/fronts-2026-11-26-r38/', '/data/reviews/fronts-2026-11-27-r48/', '/data/reviews/fronts-2026-11-27-r50/', '/data/reviews/additional-fronts-2026-11-27-r47/', '/results/review/front-review-20261126-r38/row-maps.json']], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError("Changed source preserved; choose a fresh BAOMAI_AUGMENTATION_AUDIT_BASE.")
run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
run(["git", "sparse-checkout", "set", *['/scripts/', '/metadata/model_card.json', '/models/cassette_lower_yolo26s_pose.yaml', '/data/manifest.json', '/data/labels/', '/data/candidates/front-review-2026-11-26-r37/', '/data/references/sequence-refined-reference-2026-11-18/', '/data/references/visible-front-completion-2026-11-14/', '/data/reviews/fronts-2026-11-26-r38/', '/data/reviews/fronts-2026-11-27-r48/', '/data/reviews/fronts-2026-11-27-r50/', '/data/reviews/additional-fronts-2026-11-27-r47/', '/results/review/front-review-20261126-r38/row-maps.json']], SOURCE)
SCRIPT = SOURCE / "scripts/audit_training_augmentation_r63.py"
assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == DRIVER_SHA256
def command(action, *extra):
    return [sys.executable, "-B", SCRIPT, action, "--root", SOURCE, "--work", WORK,
            "--source-revision", SOURCE_COMMIT, "--script-sha256", DRIVER_SHA256, *extra]
run(command("source-check"))
print("Fixed source:", SOURCE_COMMIT)
print("Audit outputs:", WORK)


hint: Using 'master' as the name for the initial branch. This default branch name
hint: will change to "main" in Git 3.0. To configure the initial branch name
hint: to use in all of your new repositories, which will suppress this warning,
hint: call:
hint:
hint: 	git config --global init.defaultBranch <name>
hint:
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint:
hint: 	git branch -m <name>
hint:
hint: Disable this message with "git config set advice.defaultBranchName false"
Initialized empty Git repository in /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/source/.git/
From https://github.com/Siuuuers/dl
 * branch            87b3f1dfd547bf46fef35b171aebe16d832bd1be -> FETCH_HEAD
HEAD is now at 87b3f1d Add native augmentation audit with source-row and corner lineage
{
  "commit": "87b3f1dfd547bf46fef35b171aebe16d832bd1be",
  "driver_sha256": "1bdeb8fb4df6c37ba80c6836f

## 2 Verify the native code and prepare training inputs

Existing Torch/torchvision installations are retained. If absent, the
cell installs CPU-only versions from the official PyTorch index. It then
verifies the exact five native source files controlling augmentation,
fixtures for box filtering and corner availability, archived training
labels, and every photograph hash. No checkpoint is downloaded for use
by this audit; the existing PR137 evidence archive includes model files
that are never loaded.


In [2]:
from importlib.util import find_spec
from importlib.metadata import version
if find_spec("torch") is None or find_spec("torchvision") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--index-url",
         "https://download.pytorch.org/whl/cpu", "torch==2.10.0+cpu", "torchvision==0.25.0+cpu"])
constraints = BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")))
run([sys.executable, "-m", "pip", "install", "--quiet", "-c", constraints,
     "ultralytics==8.4.182", "opencv-python==5.0.0.93", "numpy==2.4.6", "Pillow==13.0.0"])
run(command("native-check"))
sys.path.insert(0, str(SOURCE / "scripts"))
import audit_additional_front_pilot_r47_results as prior
ARCHIVE = prior.obtain_archive(CACHE / "pr137")
run(command("prepare", "--archive", ARCHIVE, "--inputs-dir", ASSETS, "--download-inputs"))
print("142 training photographs are verified. The 20 evaluation photographs are not staged.")



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
{
  "packages": {
    "ultralytics": "8.4.182",
    "torch": "2.10.0+cpu",
    "torchvision": "0.25.0+cpu",
    "opencv-python": "5.0.0.93",
    "numpy": "2.4.6",
    "Pillow": "13.0.0",
    "albumentations": null
  },
  "python": "3.12.14",
  "native_files": {
    "data/augment.py": {
      "git_blob_sha1": "14d574c52f2d92809a5c9f7d0fa1e78ec3889560",
      "sha256": "802333ea359def55e88f61d6726e35c29d28a6491110840a5e912ec1fafd1de3"
    },
    "data/base.py": {
      "git_blob_sha1": "541781d541f6f0f8fc2a2e85c57c3fcd1f327e17",
      "sha256": "4c8827598a093257a1e3218db2028f591dcd1d5ce68c63f8cdbf5741096b5f96"
    },
    "data/build.py": {
      "git_blob_sha1": "2e0ff237bc35a6f878b3a3113e6e62113830e3cd",
      "sha256": "16f13c60db87c051aec086eae82263dc7227207a8c2326c0365f013471c2609f"
    },
    "data/dataset.py": {
      "git_blob_sha1": "3f8d96688bd0bdef7a78639bc2d

## 3 Audit the actual training transforms

The audit executes the real dataset and native transformation pipeline
at the existing 1280 setting, with rectangular batches. Its observation
hooks preserve the tensors exactly, checked against uninstrumented native
calls. Native row filtering is also recomputed independently.

Each sample has a fixed, recorded seed. Both existing source-missing
points and newly cropped-out known points are tracked separately. The
source files remain unchanged. Optional Albumentations presence and its
actual transform list are recorded; an unexpected spatial transform
stops the audit rather than guessing row lineage.


In [3]:
run(command("audit", "--archive", ARCHIVE))
run(command("verify", "--archive", ARCHIVE))
result = json.loads((WORK / "results/summary.json").read_text())
print(json.dumps(result["counts"], indent=2))
print(result["interpretation"])


train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3404.4±2251.5 MB/s, size: 863.1 KB)

train: Scanning /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/dataset/labels/train... 14 images, 0 backgrounds, 0 corrupt: 9% ━─────────── 14/142 37.0it/s 0.1s<3.5s
train: Scanning /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/dataset/labels/train... 35 images, 0 backgrounds, 0 corrupt: 24% ━━╸───────── 35/142 85.7it/s 0.2s<1.2s
train: Scanning /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/dataset/labels/train... 49 images, 0 backgrounds, 0 corrupt: 34% ━━━━──────── 49/142 101.4it/s 0.3s<0.9s
train: Scanning /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/dataset/labels/train... 64 images, 0 backgrounds, 0 corrupt: 45% ━━━━━─────── 64/142 112.9it/s 0.4s<0.7s
train: Scanning /workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/dataset/labels/train... 80 images, 0 backgrounds, 0 corrupt: 56% ━━━━━━╸───── 8

## 4 Read, inspect and download

Review the original and transformed pixels side by side. Red highlights
rows removed by the native filter; orange highlights retained clipped
rows or a known corner that moved outside the canvas. These are review
categories, not a verdict that the labels are wrong. The return includes
every source-row lineage, aggregate counts, source/runtime identities,
unaugmented controls, and the small panel selection.

This diagnostic can motivate one later augmentation-policy comparison.
It cannot demonstrate that augmentation caused the detector's current
misses or that a new policy improves recognition.


In [4]:
run(command("package", "--archive", ARCHIVE))
from IPython.display import Image, Markdown, FileLink, display
display(Markdown((WORK / "results/results.md").read_text()))
for panel in result["panels"]:
    display(Markdown("**" + panel["sample_id"] + "** — " + panel["category"]))
    display(Image(filename=str(WORK / "results" / panel["path"]), width=1400))
RETURN = WORK / "baomai-augmentation-audit-r63-results.zip"
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RETURN)))
else:
    files.download(str(RETURN))


/workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/baomai-augmentation-audit-r63-results.zip


# Native augmentation and label-survival audit

This CPU diagnostic runs the existing augmentation. It does not train or score a model, change annotations, or replay historical training draws.

Three fixed passes contain 426 image presentations and 11997 source-object presentations.
Native filtering retains 11880 and removes 117; 368 retained boxes are clipped.

Of the removed presentations, 108 have zero clipped box area. The remaining 9 retain positive box area, with a largest short side of 4.047 pixels. These are box measurements, not physical object-visibility measurements.

| Source known corners | Presentations | Retained | Removed | Box clipped | Known corners outside canvas |
| --- | ---: | ---: | ---: | ---: | ---: |
| 0 | 411 | 397 | 14 | 26 | 0 |
| 1 | 1395 | 1370 | 25 | 83 | 39 |
| 2 | 10191 | 10113 | 78 | 376 | 279 |

## Interpretation

Native transformed-label arithmetic and row lineage are verified. A removed row or changed visible-point count is not by itself an error. Box area ratios do not measure physical visible material. Human image review is required to judge recognizability after cropping; this audit cannot prove which augmentation would improve a detector.

Source `v=0` means an unavailable annotation, not a negative physical existence label. A newly cropped-out known corner becomes unavailable in the transformed sample. Native internal coordinates for unavailable slots need not remain `0 0`; those coordinates are masked by the visibility flag. Canonical source labels remain unchanged.

The native box filter uses width and height > 2 pixels, area ratio > 0.10 and aspect ratio < 100. Reasons may overlap. The area ratio compares the clipped transformed axis-aligned box to the scaled source box; it is not an occlusion fraction.

## Review panels

Red marks source rows removed by the native filter; orange marks retained rows affected by clipping or lost known corners. These colors flag review cases, not annotation verdicts. Row numbers refer to the frozen training label file. Cyan/magenta dots show known left/right corner slots; unavailable points are not drawn. Panels are high-quality JPEG display copies; all measurement uses verified original images and numerical target geometry.

- [frame_0178/pass01](panels/01-frame_0178-pass01.jpg) — removed.
- [new_000002/pass01](panels/02-new_000002-pass01.jpg) — clipped_or_visibility.
- [new_000003/pass01](panels/03-new_000003-pass01.jpg) — clipped_or_visibility.
- [new_000004/pass01](panels/04-new_000004-pass01.jpg) — removed.
- [new_000005/pass01](panels/05-new_000005-pass01.jpg) — clipped_or_visibility.
- [new_000012/pass01](panels/06-new_000012-pass01.jpg) — removed.


**frame_0178/pass01** — removed

Image displayed during execution: [01-frame_0178-pass01.jpg](results/panels/01-frame_0178-pass01.jpg)  
SHA256: `71434ed7d1133572d046ebe725583cf4f5d71bec4851a2f16c4c2eadca9ca879`

**new_000002/pass01** — clipped_or_visibility

Image displayed during execution: [02-new_000002-pass01.jpg](results/panels/02-new_000002-pass01.jpg)  
SHA256: `4732a47df6ec7d73730f0daefd6d9bfda83f93a9956f56bd30d32b25cb4e199f`

... (5 outputs left out)

Image displayed during execution: [05-new_000005-pass01.jpg](results/panels/05-new_000005-pass01.jpg)  
SHA256: `79da6d69274b652e779efea54f3ad161a8ff2c0c7aaab6ed8e1e43906d3ee62d`

**new_000012/pass01** — removed

Image displayed during execution: [06-new_000012-pass01.jpg](results/panels/06-new_000012-pass01.jpg)  
SHA256: `6969b69191b4598330360e35a844e4022e3a3863c9f02736b2791530c2ebf0e0`

/workspace/scratch/f5388ec04e55/r63-augmentation-notebook-run/work/baomai-augmentation-audit-r63-results.zip